In [1]:
"""
compute_time_average.py
-----------------------
For each case folder under BASE_DIR, reads all CSV frames (e.g. Rec-XXXX_0.csv,
Rec-XXXX_1.csv, ...), computes the pixel-wise time average, and writes the result
to:
    <case_folder>/time_average/time_average.csv

The output keeps the same 9-line header as the first frame, with the Time field
updated to "Time = TIME_AVERAGED" and FrameNumber updated to the total frame count.
The numerical data is written in the same scientific-notation format as the input.

Usage
-----
    python compute_time_average.py

Edit BASE_DIR below if needed, or pass it as a command-line argument:
    python compute_time_average.py "D:/NRC/FCAI_combined/effusion_coupon/experiment/infared_data"
"""

import os
import sys
import glob
import re
import numpy as np

# ── Configuration ─────────────────────────────────────────────────────────────
BASE_DIR = r"D:\FCAI\plain_coupon\infared_data"

# Number of header lines before the blank separator line
HEADER_LINES = 9          # lines 1-9 are metadata
BLANK_LINE_INDEX = 9      # line index 9 (0-based) is blank, data starts at 10

# Output sub-folder name created inside each case folder
OUTPUT_SUBDIR = "time_average"

# Output CSV filename
OUTPUT_FILENAME = "time_average.csv"
# ──────────────────────────────────────────────────────────────────────────────


def parse_csv(filepath):
    """Return (header_lines, data_array).

    header_lines : list of raw strings (the first HEADER_LINES lines)
    data_array   : 2-D numpy float array  (rows x cols)
    """
    with open(filepath, "r") as f:
        all_lines = f.readlines()

    header = all_lines[:HEADER_LINES]          # 9 metadata lines
    # Skip the blank separator line; everything after is numeric
    data_lines = [l.strip() for l in all_lines[HEADER_LINES + 1:]
                  if l.strip()]                # drop blank / trailing lines

    data = np.array(
        [[float(v) for v in row.split(",")] for row in data_lines],
        dtype=np.float64
    )
    return header, data


def build_output_header(first_header, n_frames):
    """Modify the header from the first frame for the averaged output."""
    new_header = []
    for line in first_header:
        stripped = line.rstrip("\n").rstrip("\r")
        if stripped.startswith("Time ="):
            new_header.append("Time = TIME_AVERAGED\n")
        elif stripped.startswith("FrameNumber ="):
            new_header.append(f"FrameNumber = {n_frames} (averaged)\n")
        else:
            new_header.append(line if line.endswith("\n") else line + "\n")
    return new_header


def write_averaged_csv(out_path, header_lines, avg_data):
    """Write header + blank line + averaged data in scientific notation."""
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, "w", newline="") as f:
        # Write header
        for line in header_lines:
            f.write(line)
        # Blank separator line
        f.write("\n")
        # Write data rows
        for row in avg_data:
            f.write(",".join(f"{v:.6e}" for v in row) + "\n")


def discover_frames(case_dir):
    """Return sorted list of CSV frame paths inside case_dir.

    Looks for any *.csv files whose name matches <prefix>_<integer>.csv,
    i.e. the standard Rec-XXXX_N.csv pattern (but works with any prefix).
    """
    pattern = os.path.join(case_dir, "*.csv")
    all_csvs = glob.glob(pattern)

    frame_re = re.compile(r"^(.+)_(\d+)\.csv$", re.IGNORECASE)
    frames = []
    for p in all_csvs:
        basename = os.path.basename(p)
        m = frame_re.match(basename)
        if m:
            frames.append((int(m.group(2)), p))   # (frame_index, path)

    # Sort by frame index
    frames.sort(key=lambda x: x[0])
    return [p for _, p in frames]


def process_case(case_dir):
    """Compute the time average for one case folder and save the result."""
    frame_paths = discover_frames(case_dir)

    if not frame_paths:
        print(f"  [SKIP] No frame CSVs found in: {case_dir}")
        return

    print(f"  Processing {len(frame_paths)} frames ...")

    # Read first frame to get header and shape
    first_header, first_data = parse_csv(frame_paths[0])
    accumulator = first_data.copy()

    for fp in frame_paths[1:]:
        _, data = parse_csv(fp)
        if data.shape != accumulator.shape:
            print(f"    [WARNING] Shape mismatch in {fp}, skipping frame.")
            continue
        accumulator += data

    avg_data = accumulator / len(frame_paths)

    # Build output header
    out_header = build_output_header(first_header, len(frame_paths))

    # Write output
    out_dir = os.path.join(case_dir, OUTPUT_SUBDIR)
    out_path = os.path.join(out_dir, OUTPUT_FILENAME)
    write_averaged_csv(out_path, out_header, avg_data)
    print(f"  Saved → {out_path}")


def main():
    base = BASE_DIR

    # Normalise path separators (helps when mixing / and \ on Windows)
    base = os.path.normpath(base)
    print(f"Looking for data in:\n  {base}\n")

    if not os.path.isdir(base):
        print(f"ERROR: BASE_DIR does not exist or is not accessible:\n  {base}")
        print("  → Edit the BASE_DIR variable at the top of this script and re-run.")
        return   # use return instead of sys.exit so IPython doesn't raise SystemExit

    # Each immediate subdirectory of base is treated as a case folder
    entries = sorted(os.listdir(base))
    case_dirs = [os.path.join(base, e) for e in entries
                 if os.path.isdir(os.path.join(base, e))]

    if not case_dirs:
        print(f"No sub-folders found in {base}")
        return

    print(f"Found {len(case_dirs)} case folder(s) under:\n  {base}\n")

    for case_dir in case_dirs:
        case_name = os.path.basename(case_dir)
        # Skip previously created time_average folders if re-running
        if case_name == OUTPUT_SUBDIR:
            continue
        print(f"[{case_name}]")
        try:
            process_case(case_dir)
        except Exception as exc:
            print(f"  [ERROR] {exc}")

    print("\nDone.")


if __name__ == "__main__":
    main()

Looking for data in:
  D:\FCAI\plain_coupon\infared_data

Found 17 case folder(s) under:
  D:\FCAI\plain_coupon\infared_data

[.ipynb_checkpoints]
  [SKIP] No frame CSVs found in: D:\FCAI\plain_coupon\infared_data\.ipynb_checkpoints
[anaconda_projects]
  [SKIP] No frame CSVs found in: D:\FCAI\plain_coupon\infared_data\anaconda_projects
[cr_1400_phi_085]
  Processing 500 frames ...
  Saved → D:\FCAI\plain_coupon\infared_data\cr_1400_phi_085\time_average\time_average.csv
[cr_1400_phi_09]
  Processing 500 frames ...
  Saved → D:\FCAI\plain_coupon\infared_data\cr_1400_phi_09\time_average\time_average.csv
[cr_1400_phi_10]
  Processing 500 frames ...
  Saved → D:\FCAI\plain_coupon\infared_data\cr_1400_phi_10\time_average\time_average.csv
[cr_1400_phi_118]
  Processing 500 frames ...
  Saved → D:\FCAI\plain_coupon\infared_data\cr_1400_phi_118\time_average\time_average.csv
[cr_1400_phi_137]
  Processing 500 frames ...
  Saved → D:\FCAI\plain_coupon\infared_data\cr_1400_phi_137\time_average\tim